In [3]:
import pandas as pd
import pyrsm as rsm

In [8]:
airbnb = pd.read_csv('airbnb.csv')
blueprinty = pd.read_csv('blueprinty.csv')

In [10]:
blueprinty["age_squared"] = blueprinty["age"] ** 2
region_dummies = pd.get_dummies(blueprinty["region"], drop_first=True)
X = pd.concat([
    pd.Series(1, index=blueprinty.index, name="Intercept"),
    blueprinty[["age", "age_squared"]],
    region_dummies,
    blueprinty[["iscustomer"]]
], axis=1)

In [15]:
X.shape

(1500, 8)

In [12]:
y = blueprinty["patents"].values

In [14]:
y.shape

(1500,)

In [22]:
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from scipy.special import gammaln

def neg_poisson_regression_likelihood(beta, X, y):
    eta = np.dot(X, beta)
    eta = np.clip(eta, -20, 20)
    lambdas = np.exp(eta)
    loglik = -np.sum(lambdas) + np.dot(y, eta) - np.sum(gammaln(y + 1))
    return -loglik

# 1. 构造衍生变量
blueprinty["age_squared"] = blueprinty["age"] ** 2

# 2. 对数值变量标准化（非常关键）
from sklearn.preprocessing import StandardScaler
numeric_cols = ["age", "age_squared", "iscustomer"]
scaler = StandardScaler()
blueprinty[numeric_cols] = scaler.fit_transform(blueprinty[numeric_cols])

# 3. 构造哑变量
region_dummies = pd.get_dummies(blueprinty["region"], drop_first=True)

# 4. 拼接设计矩阵 X
X = pd.concat([
    pd.Series(1.0, index=blueprinty.index, name="Intercept"),
    blueprinty[["age", "age_squared"]],
    region_dummies,
    blueprinty[["iscustomer"]]
], axis=1)

# 5. 转换为 numpy 数组
X_names = X.columns.tolist()
X = X.astype(float).values
y = blueprinty["patents"].astype(float).values

init_beta = np.zeros(X.shape[1])

result = minimize(neg_poisson_regression_likelihood, x0=init_beta, args=(X,y), method="L-BFGS-B")
beta_hat = result.x
hessian_inv_mat = result.hess_inv.todense()  # 转换为 2D matrix
standard_errors = np.sqrt(np.diag(hessian_inv_mat))

# 打印结果表格
coef_table = pd.DataFrame({
    "Coefficient": beta_hat,
    "Std. Error": standard_errors
}, index=X_names)

print(coef_table.round(4))

             Coefficient  Std. Error
Intercept         1.2555      0.4256
age               1.0761      3.2764
age_squared      -1.1816      3.3572
Northeast         0.0292      0.5956
Northwest        -0.0176      0.7796
South             0.0565      0.7548
Southwest         0.0506      1.0325
iscustomer        0.0969      0.2682


In [26]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. 保证使用的是标准化后的 blueprinty 数据（你已经做了）

# 2. 构建公式
formula = "patents ~ age + I(age ** 2) + C(region) + iscustomer"

# 3. 拟合泊松回归模型
glm_model = smf.glm(formula=formula, data=blueprinty, family=sm.families.Poisson()).fit()

# 4. 输出结果
print(glm_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                patents   No. Observations:                 1500
Model:                            GLM   Df Residuals:                     1492
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3258.1
Date:                Fri, 02 May 2025   Deviance:                       2143.3
Time:                        15:49:16   Pearson chi2:                 2.07e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1360
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  1

In [27]:
model_std = smf.glm(
    "patents ~ age + age_squared + C(region) + iscustomer",
    data=blueprinty,  # 是标准化过的 dataframe
    family=sm.families.Poisson()
).fit()

print(model_std.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                patents   No. Observations:                 1500
Model:                            GLM   Df Residuals:                     1492
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3258.1
Date:                Fri, 02 May 2025   Deviance:                       2143.3
Time:                        15:49:56   Pearson chi2:                 2.07e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1360
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  1